# Lab 06 — OpenClaw Runtime

## Objective

Check gateway health before attempting a bounded OpenClaw CLI call.

## Introduction and concept

request enforces a timeout and rejects malformed responses. agent_turn checks gateway health before dispatching a fixed benign fixture prompt.

## Real-world use case

An optional exam assistant must handle an unavailable external runtime without hanging.

## Technology stack and dependencies

Python subprocess, JSON and optional OpenClaw CLI. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Gateway status → RPC check → CLI invocation → JSON parsing


## Step 1 — Implement the components

request enforces a timeout and rejects malformed responses. agent_turn checks gateway health before dispatching a fixed benign fixture prompt.

In [1]:
import argparse
import json
import os
import subprocess

def request(command, runner, timeout=5):
    try:
        result = runner(command, capture_output=True, text=True, timeout=timeout, check=True)
        payload = json.loads(result.stdout)
    except subprocess.TimeoutExpired as error:
        raise RuntimeError("OpenClaw command timed out") from error
    except (OSError, subprocess.CalledProcessError, json.JSONDecodeError) as error:
        raise RuntimeError("OpenClaw command failed or returned invalid JSON") from error
    if not isinstance(payload, dict):
        raise RuntimeError("OpenClaw response must be a JSON object")
    return payload

def gateway_status(executable, runner):
    return request([executable, "gateway", "status", "--json"], runner)

def agent_turn(executable, agent, runner):
    if gateway_status(executable, runner).get("rpc", {}).get("ok") is not True:
        raise RuntimeError("Gateway RPC is unhealthy")
    payload = request(
        [executable, "agent", "--agent", agent, "--message",
         "Summarize this fixture: the exam is scheduled for Friday.", "--json"], runner
    )
    if payload.get("ok") is not True:
        raise RuntimeError("Gateway reported an unsuccessful turn")
    return payload

def fixture_runner(command, **options):
    payload = {"rpc": {"ok": True}} if command[1] == "gateway" else {
        "ok": True, "summary": "The exam is scheduled for Friday."
    }
    return subprocess.CompletedProcess(command, 0, stdout=json.dumps(payload))

## Step 2 — Run sample inputs and inspect actual output

Run healthy, timeout, malformed response and unhealthy gateway fixtures. The CLI adapter receives a simulated runner.

In [2]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--live-status", action="store_true", help="Check a configured local Gateway")
    args = parser.parse_args([])
    executable = os.getenv("OPENCLAW_EXECUTABLE", "openclaw")
    if args.live_status:
        print("Live Gateway status:", gateway_status(executable, subprocess.run))
        return
    payload = agent_turn(executable, "exam-helper", fixture_runner)
    print("Simulated Gateway and CLI turn:", payload)
    assert payload["summary"].endswith("Friday.")
    def timeout_runner(command, **options):
        raise subprocess.TimeoutExpired(command, options["timeout"])
    def invalid_json_runner(command, **options):
        return subprocess.CompletedProcess(command, 0, stdout="unparseable")
    def unhealthy_runner(command, **options):
        return subprocess.CompletedProcess(command, 0, stdout='{"rpc":{"ok":false}}')
    for runner, expected in [
        (timeout_runner, "timed out"), (invalid_json_runner, "invalid JSON"),
        (unhealthy_runner, "unhealthy"),
    ]:
        try:
            agent_turn(executable, "exam-helper", runner)
        except RuntimeError as error:
            assert expected in str(error)
            print("PASS:", error)
        else:
            raise AssertionError("Expected a handled runtime failure")

main()

Simulated Gateway and CLI turn: {'ok': True, 'summary': 'The exam is scheduled for Friday.'}
PASS: OpenClaw command timed out
PASS: OpenClaw command failed or returned invalid JSON
PASS: Gateway RPC is unhealthy


## Testing and validation

Run healthy, timeout, malformed response and unhealthy gateway fixtures. The CLI adapter receives a simulated runner. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The captured output uses an injected subprocess fixture. The optional live status command requires a locally installed CLI and configured Gateway. The notebook does not invoke a live agent.

## How this relates to ExamPilot

Implemented optional adapter. Compare the command shapes and error handling in backend/app/openclaw/runtime.py and backend/app/graph/openclaw_node.py.

## Conclusion

Check gateway health before attempting a bounded OpenClaw CLI call. Use the trace and assertions to explain which decisions the workflow makes.